# Previsão de Preço de Diamantes

In [ ]:
!pip install tensorflow pandas seaborn scikit-learn joblib numpy matplotlib -q

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Carregamento e Exploração dos Dados

In [ ]:
try:
    df = pd.read_csv('diamonds.csv')
except:
    df = sns.load_dataset('diamonds')

df.head(10)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Análise Exploratória

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Preço ($)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição dos Preços')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', label=f"Média: ${df['price'].mean():,.0f}")
axes[0].axvline(df['price'].median(), color='green', linestyle='--', label=f"Mediana: ${df['price'].median():,.0f}")
axes[0].legend()

axes[1].boxplot(df['price'], vert=True)
axes[1].set_ylabel('Preço ($)')
axes[1].set_title('Boxplot dos Preços')

plt.tight_layout()
plt.show()

### Matriz de Correlação

In [ ]:
numerical_cols_corr = ['carat', 'depth', 'table', 'x', 'y', 'z', 'price']
correlation_matrix = df[numerical_cols_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

### Carat vs Preço

In [ ]:
plt.figure(figsize=(12, 6))
sample = df.sample(5000, random_state=42)
scatter = plt.scatter(sample['carat'], sample['price'], alpha=0.5, c=sample['price'], cmap='viridis', s=15)
plt.colorbar(scatter, label='Preço ($)')
plt.xlabel('Quilates (Carat)')
plt.ylabel('Preço ($)')
plt.title('Carat vs Preço')
plt.tight_layout()
plt.show()

## 3. Pré-processamento

In [ ]:
y = df['price']
X = df.drop('price', axis=1)

categorical_cols = ['cut', 'color', 'clarity']
numerical_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

input_shape = X_train_processed.shape[1]

## 4. Treinamento dos Modelos

In [ ]:
def create_model_1(input_shape):
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=[input_shape]),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

def create_model_2(input_shape):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=[input_shape]),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

In [ ]:
model1 = create_model_1(input_shape)
history1 = model1.fit(X_train_processed, y_train, validation_split=0.2, batch_size=32, epochs=50, verbose=1)

model2 = create_model_2(input_shape)
history2 = model2.fit(X_train_processed, y_train, validation_split=0.2, batch_size=32, epochs=50, verbose=1)

### Histórico de Treinamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history1.history['mae'], label='Treino', color='#3498db', linewidth=2)
axes[0].plot(history1.history['val_mae'], label='Validação', color='#e74c3c', linewidth=2)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MAE ($)')
axes[0].set_title('Modelo 1')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history2.history['mae'], label='Treino', color='#3498db', linewidth=2)
axes[1].plot(history2.history['val_mae'], label='Validação', color='#e74c3c', linewidth=2)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE ($)')
axes[1].set_title('Modelo 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Avaliação

In [ ]:
loss1, mae1 = model1.evaluate(X_test_processed, y_test, verbose=0)
loss2, mae2 = model2.evaluate(X_test_processed, y_test, verbose=0)

pred1 = model1.predict(X_test_processed, verbose=0).flatten()
pred2 = model2.predict(X_test_processed, verbose=0).flatten()
pred_voting = (pred1 + pred2) / 2

mae_voting = np.mean(np.abs(y_test - pred_voting))

results = pd.DataFrame({
    'Modelo': ['Modelo 1', 'Modelo 2', 'Voting Ensemble'],
    'MAE ($)': [mae1, mae2, mae_voting]
})
results

### Previsão vs Real

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sample_size = 1000
indices = np.random.choice(len(y_test), sample_size, replace=False)
y_sample = y_test.iloc[indices].values

max_val = max(y_sample.max(), pred1[indices].max())

axes[0].scatter(y_sample, pred1[indices], alpha=0.5, s=10, c='#3498db')
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2)
axes[0].set_xlabel('Preço Real ($)')
axes[0].set_ylabel('Preço Previsto ($)')
axes[0].set_title(f'Modelo 1 (MAE: ${mae1:,.0f})')

axes[1].scatter(y_sample, pred2[indices], alpha=0.5, s=10, c='#27ae60')
axes[1].plot([0, max_val], [0, max_val], 'r--', lw=2)
axes[1].set_xlabel('Preço Real ($)')
axes[1].set_ylabel('Preço Previsto ($)')
axes[1].set_title(f'Modelo 2 (MAE: ${mae2:,.0f})')

axes[2].scatter(y_sample, pred_voting[indices], alpha=0.5, s=10, c='#9b59b6')
axes[2].plot([0, max_val], [0, max_val], 'r--', lw=2)
axes[2].set_xlabel('Preço Real ($)')
axes[2].set_ylabel('Preço Previsto ($)')
axes[2].set_title(f'Voting Ensemble (MAE: ${mae_voting:,.0f})')

plt.tight_layout()
plt.show()

### Comparação de MAE

In [ ]:
models_names = ['Modelo 1', 'Modelo 2', 'Voting']
maes = [mae1, mae2, mae_voting]
colors = ['#3498db', '#27ae60', '#9b59b6']

plt.figure(figsize=(8, 5))
bars = plt.bar(models_names, maes, color=colors, edgecolor='white', linewidth=2)
plt.ylabel('MAE ($)')
plt.title('Comparação de MAE')

for bar, mae in zip(bars, maes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
             f'${mae:,.0f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Salvar Modelos

In [ ]:
model1.save("model1.keras")
model2.save("model2.keras")
joblib.dump(preprocessor, "preprocessor.joblib")

import zipfile
with zipfile.ZipFile("models.zip", "w") as z:
    z.write("model1.keras")
    z.write("model2.keras")
    z.write("preprocessor.joblib")

from google.colab import files
files.download("models.zip")

## 7. Teste

In [ ]:
loaded_model1 = keras.models.load_model("model1.keras")
loaded_model2 = keras.models.load_model("model2.keras")
loaded_preprocessor = joblib.load("preprocessor.joblib")

test_diamond = pd.DataFrame([{
    'carat': 1.0,
    'cut': 'Ideal',
    'color': 'G',
    'clarity': 'VS1',
    'depth': 61.5,
    'table': 55.0,
    'x': 6.5,
    'y': 6.5,
    'z': 4.0
}])

test_processed = loaded_preprocessor.transform(test_diamond)
pred1_test = loaded_model1.predict(test_processed, verbose=0)[0][0]
pred2_test = loaded_model2.predict(test_processed, verbose=0)[0][0]
pred_voting_test = (pred1_test + pred2_test) / 2

pd.DataFrame({
    'Modelo': ['Modelo 1', 'Modelo 2', 'Voting'],
    'Previsão ($)': [pred1_test, pred2_test, pred_voting_test]
})